## import libraries

In [4]:
%run ./utils.py

ModuleNotFoundError: No module named 'numerize'

In [9]:
# import libraries
import os
import sys
import subprocess
from string import punctuation
import unicodedata

import pandas as pd
import numpy as np

import time
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta

# from numerize.numerize import numerize
from scipy import stats

import requests

import json
import multiprocessing
import re
from unidecode import unidecode
from operator import add

from functools import reduce
from itertools import chain

from pyarrow import fs
import pyarrow.parquet as pq

import warnings
warnings.filterwarnings('ignore')

os.environ['HTTP_PROXY'] = "http://proxy.hcm.fpt.vn:80"
os.environ['HTTPS_PROXY'] = "http://proxy.hcm.fpt.vn:80"

os.environ['HADOOP_CONF_DIR'] = "/etc/hadoop/conf/"
os.environ['JAVA_HOME'] = "/usr/jdk64/jdk1.8.0_112"
os.environ['HADOOP_HOME'] = "/usr/hdp/3.1.0.0-78/hadoop"
os.environ['ARROW_LIBHDFS_DIR'] = "/usr/hdp/3.1.0.0-78/usr/lib/"
os.environ['CLASSPATH'] = subprocess.check_output("$HADOOP_HOME/bin/hadoop classpath --glob", shell=True).decode('utf-8')

hdfs = fs.HadoopFileSystem(host="hdfs://hdfs-cluster.datalake.bigdata.local", port=8020)

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', -1)

# import pyspark
from pyspark import SparkConf
from pyspark.sql import SparkSession

from pyspark.sql import functions as F

from pyspark.sql import DataFrame
from typing import Iterable

from pyspark.sql.window import Window

def initial_spark(spark_name='demo', mode='local[16]', driver_memory='20g', max_worker=1):
    
    # for yarn support
    os.environ['SPARK_HOME']="/opt/spark/spark-3.0.2-bin-hadoop2.7/"
    # os.environ[
    #     "SPARK_HOME"
    # ] = "/opt/spark/spark-3.0.2-bin-hadoop2.7-lineage/"  # Change SPARK_HOME to lineage
    os.environ['JAVA_HOME']="/usr/jdk64/jdk1.8.0_112/"
    os.environ['SPARK_CONF_DIR']= ''
    
    conf = SparkConf()
    
    # config spark application name
    conf.setAppName(spark_name)

    # config location for spark finding metadata from hive metadata server
    conf.set("hive.metastore.uris", "thrift://master01-dc9c14u40.bigdata.local:9083,thrift://master02-dc9c14u41.bigdata.local:9083")
    conf.set("spark.sql.hive.metastore.jars", "/opt/spark/spark-3.0.2-bin-hadoop2.7/*")

    # config in-memory columnar data format that is used in Spark to efficiently transfer data between JVM and Python processes
    conf.set("spark.kryoserializer.buffer.max", "2000")
    conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    conf.set("spark.sql.execution.arrow.enabled", "true")
    conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")

    # config spark driver memory
    conf.set("spark.driver.memory", driver_memory)
    conf.set('spark.driver.maxResultSize', driver_memory)

    #set metastore.client.capability.check to false
    conf.set("hive.metastore.client.capability.check", "false")
    conf.set("spark.port.maxRetries", 100)
    conf.set("spark.jars","hdfs:///shared/jars/hotpot_2.12-0.0.3.jar")
    conf.set("spark.sql.redaction.string.regex",".{22}==")
    
    # config timezone
    conf.get('spark.sql.session.timeZone')
    conf.get("hive.metastore.uris")
    
    # config show pandas dataframe format on notebook
    conf.set("spark.sql.repl.eagerEval.enabled",True)
    conf.set("spark.sql.repl.eagerEval.truncate",200)
    conf.get("hive.metastore.uris")
    
    # config calendar
    conf.set("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "LEGACY")
    conf.set("spark.sql.legacy.parquet.int96RebaseModeInWrite", "LEGACY")

    # config yarn
    if mode == 'yarn':
        conf.set("spark.executor.memory", "12g")
        conf.set("spark.executor.cores", "4")
        conf.set("spark.dynamicAllocation.enabled", True)
        conf.set("spark.dynamicAllocation.initialExecutors", 1)
        conf.set("spark.dynamicAllocation.shuffleTracking.enabled", True)
        conf.set("spark.dynamicAllocation.minExecutors", 1)
        conf.set("spark.dynamicAllocation.maxExecutors", max_worker)
        conf.set("spark.dynamicAllocation.shuffleTracking.timeout", "3m")
        os.environ['PYSPARK_DRIVER_PYTHON'] = 'python'
        os.environ['PYSPARK_PYTHON'] = './environment/bin/python'
        conf.set("spark.yarn.dist.archives", "hdfs:/shared/envs/trinhlk2-python37_env.tar.gz#environment")
        
    # config checkpoint
    conf.set("spark.cleaner.referenceTracking.cleanCheckpoints", "true")
    
    # config maxSize
    conf.set("spark.rpc.message.maxSize", 512)
    
    # initial spark
    spark = SparkSession.builder.config(conf=conf).master(mode).enableHiveSupport().getOrCreate()
    
    # decrypt
    spark._jvm.vn.fpt.insights.utils.Helper.registerUdf("fdecrypt","fdecrypt")
    spark.sql("use ftel_dwh_isc")
    
    # set checkpoint dir
    spark.sparkContext.setCheckpointDir('/data/fpt/ftel/cads/dep_solution/sa/dev/checkpoint')
    
    # set log level
    spark.sparkContext.setLogLevel("ERROR")
    
    return spark

def rename_columns(df, columns):
    if isinstance(columns, dict):
        return df.select(*[F.col(col_name).alias(columns.get(col_name, col_name)) for col_name in df.columns])
    else:
        raise ValueError("'columns' should be a dict")

ModuleNotFoundError: No module named 'unidecode'

In [6]:
!pip install numerize

Looking in indexes: https://repo.cads.live/python/repository/pypi-group/simple

[notice] A new release of pip is available: 23.3.2 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [8]:
spark = initial_spark(mode = 'yarn', driver_memory='10g', max_worker=8)
# spark = initial_spark()

# import spark_sdk as ss
# spark = ss.PySpark(
#     yarn=True, driver_memory="4G", num_executors=4, executor_memory="16G"
# ).spark

NameError: name 'initial_spark' is not defined

In [1]:
import os
import subprocess

from pyarrow import fs

os.environ["HADOOP_CONF_DIR"] = "/etc/hadoop/conf/"
os.environ["JAVA_HOME"] = "/usr/jdk64/jdk1.8.0_112"
os.environ["HADOOP_HOME"] = "/usr/hdp/3.1.0.0-78/hadoop"
os.environ["ARROW_LIBHDFS_DIR"] = "/usr/hdp/3.1.0.0-78/usr/lib/"
os.environ["CLASSPATH"] = subprocess.check_output(
    "$HADOOP_HOME/bin/hadoop classpath --glob", shell=True
).decode("utf-8")

hdfs = fs.HadoopFileSystem(
    host="hdfs://hdfs-cluster.datalake.bigdata.local", port=8020
)

import pandas as pd
# config for pyspark
from pyspark import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os
from pyspark.sql.functions import split, when, col,lag
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, col, when
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, col, expr, lower
from pyspark.sql.functions import desc
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType




2024-08-16 13:23:34,552 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Initial Spark

In [2]:
def initial_spark(
        spark_name="spark", mode="local[16]", driver_memory="20g", max_worker=1
):
    # for yarn support
    os.environ["SPARK_HOME"] = "/opt/spark/spark-3.0.2-bin-hadoop2.7/"
    os.environ["JAVA_HOME"] = "/usr/jdk64/jdk1.8.0_112/"
    os.environ["SPARK_CONF_DIR"] = ""

    conf = SparkConf()

    # config spark application name
    conf.setAppName(spark_name)

    # config location for spark finding metadata from hive metadata server
    conf.set(
        "hive.metastore.uris",
        "thrift://master01-dc9c14u40.bigdata.local:9083,thrift://master02-dc9c14u41.bigdata.local:9083",
    )
    conf.set(
        "spark.sql.hive.metastore.jars",
        "/opt/spark/spark-3.0.2-bin-hadoop2.7/*",
    )

    # config in-memory columnar data format that is used in Spark to efficiently transfer data between JVM and Python processes
    conf.set("spark.kryoserializer.buffer.max", "2000")
    conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    conf.set("spark.sql.execution.arrow.enabled", "true")
    conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "10000")

    # config spark driver memory
    conf.set("spark.driver.memory", driver_memory)
    conf.set("spark.driver.maxResultSize", driver_memory)

    # set metastore.client.capability.check to false
    conf.set("hive.metastore.client.capability.check", "false")
    conf.set("spark.port.maxRetries", 100)
    conf.set("spark.jars", "hdfs:///shared/jars/hotpot_2.12-0.0.3.jar")
    conf.set("spark.sql.redaction.string.regex", ".{22}==")

    # config timezone
    conf.get("spark.sql.session.timeZone")
    conf.get("hive.metastore.uris")

    # config show pandas dataframe format on notebook
    conf.set("spark.sql.repl.eagerEval.enabled", True)
    conf.set("spark.sql.repl.eagerEval.truncate", 200)
    conf.get("hive.metastore.uris")

    # config calendar
    conf.set("spark.sql.legacy.parquet.datetimeRebaseModeInRead", "LEGACY")
    conf.set("spark.sql.legacy.parquet.int96RebaseModeInWrite", "LEGACY")

    # config yarn
    if mode == "yarn":
        conf.set("spark.executor.memory", "16g")
        conf.set("spark.executor.cores", "4")
        conf.set("spark.dynamicAllocation.enabled", True)
        conf.set("spark.dynamicAllocation.initialExecutors", 1)
        conf.set("spark.dynamicAllocation.shuffleTracking.enabled", True)
        conf.set("spark.dynamicAllocation.minExecutors", 1)
        conf.set("spark.dynamicAllocation.maxExecutors", max_worker)
        conf.set("spark.dynamicAllocation.shuffleTracking.timeout", "3m")
        conf.set("spark.executor.memoryOverhead", "8g")

    # initial spark
    spark = (
        SparkSession.builder.config(conf=conf)
        .master(mode)
        .enableHiveSupport()
        .getOrCreate()
    )

    # decrypt
    spark._jvm.vn.fpt.insights.utils.Helper.registerUdf("fdecrypt", "fdecrypt")
    spark.sql("use ftel_dwh_isc")

    # set log level
    spark.sparkContext.setLogLevel("ERROR")

    return spark


spark = initial_spark(
    spark_name="spark-hoanglx9", mode="yarn", driver_memory="6g", max_worker=6
)

# spark = initial_spark()

2024-08-16 13:23:44,852 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
2024-08-16 13:23:49,044 WARN util.Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
2024-08-16 13:23:49,045 WARN util.Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
2024-08-16 13:23:49,046 WARN util.Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
2024-08-16 13:23:49,046 WARN util.Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
2024-08-16 13:23:49,047 WARN util.Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
2024-08-16 13:23:49,677 WARN util.Utils: spark.executor.instances less than spark.dynamicAllocation.minExecutors is invalid, ignoring its setting, please update your 

## Load data

In [4]:
# script_df = (
#     spark.read.parquet('/data/fpt/fdp/cdp/dwh/stag_user_behavior.parquet')
#     .filter(F.col('d') >= datetime(2024, 8, 15))
#     .select(*[
#         'timestamp',
#         'ip',
#         'cdp_id',
#         'event',
#         'type',
#         'user_agent',
#         'item_title',
#         'item_id',
#         'category_id',
#         'category_name',
#         'item_cate',
#         'subcategory_id',
#         'subcategory_name',
#         'path',
#         'referrer',
#         'search',
#         'title',
#         'url',
#         'campaign_content',
#         'campaign_medium',
#         'campaign_name',
#         'campaign_source',
#         'campaign_term',
#         'd',
#         # 'cookies',
#         # 'last_browsed_items',
#         # 'total_cart_amount',
#         # 'custom_attributes',
#         # 'buy_installment_button',
#         # 'user_date'
#        ]
#     )
# )
# (
#     script_df
#     .write.partitionBy("d")
#     .mode("overwrite")
#     .option("partitionOverwriteMode", "dynamic")
#     .parquet("/data/fpt/ftel/cads/dep_solution/user/trinhlk2/script_fshop.parquet")
# )

In [4]:
from pyspark.sql.functions import *

In [3]:
user = "hoanglx9"

In [4]:
df_filter_1 = spark.read.parquet('/data/fpt/ftel/cads/dep_solution/user/trinhlk2/script_fshop.parquet')

## Preprocessing

In [6]:
# Convert the "path" column to lowercase to ensure uniformity in string comparisons and operations
df_filter_1 = df_filter_1.withColumn("path", lower(col("path")))

# Remove trailing slashes from the "path" column to standardize the format
df_filter_1 = df_filter_1.withColumn(
    "path",
    when(col("path").endswith("/"), expr("substring(path, 1, length(path) - 1)")).otherwise(col("path"))
)

# Split the "path" column by the '/' character and extract the first element (second part of the path)
df_filter_1 = df_filter_1.withColumn('path_1', split(df_filter_1['path'], '/').getItem(1))

# Extract the second element (third part of the path)
df_filter_1 = df_filter_1.withColumn('path_2', split(df_filter_1['path'], '/').getItem(2))

# Extract the third element (fourth part of the path)
df_filter_1 = df_filter_1.withColumn('path_3', split(df_filter_1['path'], '/').getItem(3))

# Extract the fourth element (fifth part of the path)
df_filter_1 = df_filter_1.withColumn('path_4', split(df_filter_1['path'], '/').getItem(4))

# Extract the fifth element (sixth part of the path)
df_filter_1 = df_filter_1.withColumn('path_5', split(df_filter_1['path'], '/').getItem(5))


In [7]:
# Sort the DataFrame by the 'timestamp' column and remove duplicate rows based on 'cdp_id' and 'timestamp' columns,
# keeping the first occurrence of each unique combination (after ordering by 'timestamp').
result_df = df_filter_1.orderBy('timestamp').dropDuplicates(['cdp_id', 'timestamp'])

In [8]:
result_df.count()

101096544

### Label actions

In [9]:
# Create a new column "isInstallment" that flags rows where "path_4" equals "tra-gop" (indicating an installment)
# with a value of 1, and 0 otherwise.
result_df = result_df.withColumn("isInstallment", when(col("path_4") == "tra-gop", 1).otherwise(0))

# Define a window specification to partition data by "path_1", allowing for aggregation within each partition.
window_spec_7 = Window().partitionBy("path_1")

# Create a new column "total_isInstallment" that sums the "isInstallment" values within each "path_1" partition.
result_df = result_df.withColumn("total_isInstallment", sum("isInstallment").over(window_spec_7))

# Define another window specification to partition data by "path_2", allowing for aggregation within each partition.
window_spec_8 = Window().partitionBy("path_2")

# Create a new column "total_isInstallment_1" that sums the "isInstallment" values within each "path_2" partition.
result_df = result_df.withColumn("total_isInstallment_1", sum("isInstallment").over(window_spec_8))

# Create a new column "action" based on several conditions:
# - If "path_4" equals 'tra-gop', the action is 'installment'.
# - If "type" is 'page' and "total_isInstallment_1" is greater than 0, the action is 'viewPage'.
# - If "type" is 'page' and "total_isInstallment" is greater than 0, the action is 'findProduct'.
# - If none of the above conditions are met, the action is set to None.
result_df = result_df.withColumn("action",
                                 when((col("path_4") == 'tra-gop'), 'installment') \
                                 .when((col("type") == 'page') & (col("total_isInstallment_1") > 0), 'viewPage') \
                                 .when((col("type") == 'page') & (col("total_isInstallment") > 0), 'findProduct') \
                                 .otherwise(None)
                                 )


In [10]:
# Create a new column "isTrack" that flags rows where "type" equals "track" with a value of 1, and 0 otherwise.
result_df = result_df.withColumn("isTrack", when(col("type") == "track", 1).otherwise(0))

# Define a window specification to partition data by the "path" column.
window_spec = Window().partitionBy("path")

# Create a new column "total_isTrack" that sums the "isTrack" values within each "path" partition.
result_df = result_df.withColumn("total_isTrack", sum("isTrack").over(window_spec))

# Define another window specification to partition data by the "path_1" column.
window_spec_1 = Window().partitionBy("path_1")

# Create a new column "total_isTrack_1" that sums the "isTrack" values within each "path_1" partition.
result_df = result_df.withColumn("total_isTrack_1", sum("isTrack").over(window_spec_1))

# Define another window specification to partition data by the "path_2" column.
window_spec_2 = Window().partitionBy("path_2")

# Create a new column "total_isTrack_2" that sums the "isTrack" values within each "path_2" partition.
result_df = result_df.withColumn("total_isTrack_2", sum("isTrack").over(window_spec_2))

# Drop the original "isTrack" column since it's no longer needed after the aggregations.
result_df = result_df.drop("isTrack")

# Update the "action" column based on several conditions:
# - If "type" is 'track' and "event" is 'view', set the action to 'viewDetail'.
# - If "type" is 'track' and "event" is 'add_to_cart', set the action to 'addToCart'.
# - If "type" is 'track' and "event" is 'purchase', set the action to 'addToCart'.
# - If "type" is 'page' and "total_isTrack" is greater than 0, set the action to 'viewPage'.
# - If "type" is 'page' and "total_isTrack_1" is greater than 0, set the action to 'findProduct'.
# - If "type" is 'page' and "total_isTrack_2" is greater than 0, set the action to 'findProduct'.
# - If none of the above conditions are met, keep the original value of the "action" column.
result_df = result_df.withColumn("action",
                                 when((col("type") == 'track') & (col("event") == 'view'), 'viewDetail')
                                 .when((col("type") == 'track') & (col("event") == 'add_to_cart'), 'addToCart')
                                 .when((col("type") == 'track') & (col("event") == 'purchase'), 'addToCart')
                                 .when((col("type") == 'page') & (col("total_isTrack") > 0), 'viewPage')
                                 .when((col("type") == 'page') & (col("total_isTrack_1") > 0), 'findProduct')
                                 .when((col("type") == 'page') & (col("total_isTrack_2") > 0), 'findProduct')
                                 .otherwise(col('action'))
                                 )


In [11]:
# Define multiple conditions and their corresponding new values to categorize actions based on the URL path or title.

# Condition 1: If "path_1" equals "tin-tuc" and "path_2" is not "tin-khuyen-mai", set the action to "viewNews".
condition1 = (
        (col("path_1") == "tin-tuc") & ~(col('path_2') == 'tin-khuyen-mai')
)
new_value1 = "viewNews"

# Condition 2: If "path_1" is in the list of support-related paths, set the action to "support".
condition2 = (
    col("path_1").isin("ho-tro", "lien-he", "huong-dan", "tos", "cua-hang")
)
new_value2 = "support"

# Condition 3: If "path_1" equals "tim-kiem", set the action to "lookup".
condition3 = (
        col("path_1") == "tim-kiem"
)
new_value3 = "lookup"

# Condition 4: If "path_1" or "path_2" relates to promotions, set the action to "checkPromo".
condition4 = (
        (col("path_1") == "ctkm")
        | (col("path_2") == "tin-khuyen-mai")
        | (col("path_1") == "khuyen-mai")
)
new_value4 = "checkPromo"

# Condition 5: If "path_1" equals "so-sanh-san-pham", set the action to "compareProduct".
condition5 = (col("path_1") == "so-sanh-san-pham")
new_value5 = "compareProduct"

# Condition 6: If "path_1" indicates a cart-related path or the title is "Giỏ hàng" or "Cart", set the action to "checkCart".
condition6 = (
        (col("path_1") == "cart") |
        (col("path_1") == "gio-hang") |
        (col("path_1") == "gio-hang-v2") |
        (col('title') == 'Giỏ hàng') |
        (col('title') == 'Cart')
)
new_value6 = "checkCart"

# Condition 7: If the URL is the homepage, set the action to "homePage".
condition7 = (col("url") == "https://fptshop.com.vn/")
new_value7 = "homePage"

# Condition 8: If "path_1" or "path_4" indicates an installment-related page or the title is "Mua trả góp", set the action to "installment".
condition8 = (
        (col("path_1") == 'tra-gop') |
        (col("path_4") == 'tra-gop') |
        (col("title") == 'Mua trả góp')
)
new_value8 = "installment"

# Condition 9: If "path_1" equals "tai-khoan" and "path_2" is not "don-hang-cua-toi" or the title is "Thông tin tài khoản", set the action to "checkAccount".
condition9 = (
        ((col("path_1") == 'tai-khoan') & ~(col('path_2') == 'don-hang-cua-toi')) |
        (col("title") == 'Thông tin tài khoản')
)
new_value9 = "checkAccount"

# Condition 10: If "path_1" relates to services like "dich-vu", "phan-mem", or "sim-so-dep", set the action to "service".
condition10 = (
        (col("path_1") == 'dich-vu') |
        (col("path_1") == 'phan-mem') |
        (col("path_1") == 'sim-so-dep')
)
new_value10 = "service"

# Condition 11: If "path_1" equals "thu-cu-doi-moi", set the action to "upgrade".
condition11 = (col("path_1") == 'thu-cu-doi-moi')
new_value11 = "upgrade"

# Condition 12: If "path_1" equals "404", "Error", or "undefined", set the action to "error".
condition12 = ((col("path_1") == '404') | (col("path_1") == 'Error') | (col("path_1") == 'undefined'))
new_value12 = "error"

# Condition 13: If "path" contains "landing" but "path_1" is not "tin-tuc", set the action to "landing".
condition13 = (
        (col('path').contains('landing')) &
        ~(col('path_1') == 'tin-tuc')
)
new_value13 = "landing"

# Condition 14: If "path_1" indicates a payment-related page, set the action to "payment".
condition14 = (
    (col("path_1").isin('zalopay', 'vnpay', 'kredivo', 'thanh-toan-onlinev2', 'thanh-toan-online'))
)
new_value14 = "payment"

# Condition 15: If the URL contains "thanh-cong" and "path_1" is not "tin-tuc", set the action to "purchase".
condition15 = (
        (col("url").contains('thanh-cong')) &
        ~(col('path_1') == 'tin-tuc')
)
new_value15 = "purchase"

# Condition 16: If "path_1" equals "trackingorder", the title is "Đơn hàng của tôi", or "path_2" equals "don-hang-cua-toi", set the action to "trackingOrder".
condition16 = (
        (col("path_1") == 'trackingorder') | (col("title") == 'Đơn hàng của tôi') | (
        col('path_2') == 'don-hang-cua-toi'))
new_value16 = "trackingOrder"

# Condition 17: If "path_1" equals "kiem-tra-bao-hanh", set the action to "checkWarranty".
condition17 = col('path_1') == 'kiem-tra-bao-hanh'
new_value17 = 'checkWarranty'

# Condition 18: If "path_1" equals "tich-diem-doi-qua", set the action to "checkLoyalty".
condition18 = col('path_1') == 'tich-diem-doi-qua'
new_value18 = 'checkLoyalty'

# Update the "action" column based on the defined conditions and corresponding new values.
# If none of the conditions are met, keep the original value of the "action" column.
result_df = result_df.withColumn("action",
                                 when(condition1, new_value1)
                                 .when(condition2, new_value2)
                                 .when(condition3, new_value3)
                                 .when(condition4, new_value4)
                                 .when(condition5, new_value5)
                                 .when(condition6, new_value6)
                                 .when(condition7, new_value7)
                                 .when(condition8, new_value8)
                                 .when(condition9, new_value9)
                                 .when(condition10, new_value10)
                                 .when(condition11, new_value11)
                                 .when(condition12, new_value12)
                                 .when(condition13, new_value13)
                                 .when(condition14, new_value14)
                                 .when(condition15, new_value15)
                                 .when(condition16, new_value16)
                                 .when(condition17, new_value17)
                                 .when(condition18, new_value18)
                                 .otherwise(col('action'))  # Keep the original value if none of the conditions match
                                 )


In [12]:
# Define a condition to check if "path_2" equals "don-hang-cua-toi" and set the action to "trackingOrder".
condition19 = (col('path_2') == 'don-hang-cua-toi')
new_value19 = 'trackingOrder'

# Update the "action" column based on condition19:
# - If "path_2" equals "don-hang-cua-toi", set the action to "trackingOrder".
# - If the condition is not met, keep the original value of the "action" column.
result_df = result_df.withColumn("action",
                                 when(condition19, new_value19)
                                 .otherwise(col('action')))


In [13]:
# Define condition_17: If "path_1" belongs to a list of specified product categories (e.g., 'apple', 'xiaomi', etc.),
# set the action to "findProduct".
condition_17 = (
    (col("path_1").isin(['apple', 'xiaomi', 'samsung', 'dien-gia-dung']))
)
new_value_17 = "findProduct"

# Define condition_18: If "path_1" equals 'may-doi-tra', set the action to "viewOldProduct".
condition_18 = (
    (col("path_1").isin(['may-doi-tra']))
)
new_value_18 = "viewOldProduct"

# Define condition_19: If the "title" contains 'Giỏ hàng', set the action to "checkCart".
condition_19 = (
    (col("title").contains('Giỏ hàng'))
)
new_value_19 = "checkCart"

# Update the "action" column based on conditions 17, 18, and 19:
# - If condition_17 is met, set the action to "findProduct".
# - If condition_18 is met, set the action to "viewOldProduct".
# - If condition_19 is met, set the action to "checkCart".
# - If none of the conditions are met, keep the original value of the "action" column.
result_df = result_df.withColumn("action",
                                 when(condition_17, new_value_17)
                                 .when(condition_18, new_value_18)
                                 .when(condition_19, new_value_19)
                                 .otherwise(col('action'))  # Keep the original value if none of the conditions match
                                 )

# Define condition_20: If "action" is 'findProduct', "path_1" equals 'linh-kien', and "path_3" is null,
# set the action to "findProduct".
condition_20 = (
        (col("action") == 'findProduct') &
        (col("path_1") == 'linh-kien') &
        (col("path_3").isNull())
)
new_value_20 = "findProduct"

# Define condition_21: If "action" is 'findProduct', "path_1" equals 'linh-kien', and "path_3" is not null,
# set the action to "viewPage".
condition_21 = (
        (col("action") == 'findProduct') &
        (col("path_1") == 'linh-kien') &
        (col("path_3").isNotNull())
)

new_value_21 = "viewPage"

# Update the "action" column based on conditions 20 and 21:
# - If condition_20 is met, set the action to "findProduct".
# - If condition_21 is met, set the action to "viewPage".
# - If none of the conditions are met, keep the original value of the "action" column.
result_df = result_df.withColumn("action",
                                 when(condition_20, new_value_20)
                                 .when(condition_21, new_value_21)
                                 .otherwise(col('action'))  # Keep the original value if none of the conditions match
                                 )

# Write the final DataFrame to a partitioned Parquet file.
(
    result_df
    .write.partitionBy("d")  # Partition the data by column "d"
    .mode("overwrite")  # Overwrite the existing data
    .option("partitionOverwriteMode", "dynamic")  # Allow dynamic partition overwrite
    .parquet(
        f"/data/fpt/ftel/cads/dep_solution/user/{user}/fshop_script_action_tmp.parquet"  # File path to save the data
    )
)


### Adding product category

In [5]:
# Read the Parquet file into a DataFrame
result_df = spark.read.parquet(f'/data/fpt/ftel/cads/dep_solution/user/{user}/fshop_script_action_tmp.parquet')

# Extract and assign 'product' based on the 'title' column for actions 'viewPage' and 'viewDetail'
result_df = result_df.withColumn('product',
                                 F.when(((col('action') == 'viewPage') | (col('action') == 'viewDetail')),
                                        F.split(col('title'), '\|').getItem(0))
                                 .otherwise(None))  # Set to None if the condition is not met

# Extract and assign 'type_product' based on the 'path' column for actions 'viewPage' and 'viewDetail'
result_df = result_df.withColumn('type_product',
                                 F.when(((col('action') == 'viewPage') | (col('action') == 'viewDetail')),
                                        F.split(col('path'), '/').getItem(1))
                                 .otherwise(None))  # Set to None if the condition is not met

# Forward fill the 'product' column with the last non-null value for each 'cdp_id', ordered by 'timestamp'
result_df = result_df.withColumn("product", 
                                 F.last("product", ignorenulls=True).over(
                                     Window().partitionBy('cdp_id')
                                            .orderBy("timestamp")
                                            .rowsBetween(Window.unboundedPreceding, 0)
                                 ))

# Forward fill the 'type_product' column with the last non-null value for each 'cdp_id', ordered by 'timestamp'
result_df = result_df.withColumn("type_product", 
                                 F.last("type_product", ignorenulls=True).over(
                                     Window().partitionBy('cdp_id')
                                            .orderBy("timestamp")
                                            .rowsBetween(Window.unboundedPreceding, 0)
                                 ))


### Calculate duration and session

In [6]:
# Convert the 'timestamp' column to a date type (if needed)
# df = df.withColumn("timestamp", col("timestamp").cast("date"))

# Define a window specification to partition by 'cdp_id' and order by 'timestamp'
window_spec_5 = Window().partitionBy("cdp_id").orderBy("timestamp")

# Calculate the previous timestamp within each partitioned group
result_df = result_df.withColumn("prev_timestamp", lag("timestamp").over(window_spec_5))

# Calculate the duration between the current and previous timestamp in seconds
result_df = result_df.withColumn("duration",
                                 (col("timestamp").cast("long") - col("prev_timestamp").cast("long")))

# Drop the 'prev_timestamp' column as it is no longer needed
result_df = result_df.drop("prev_timestamp")

# Define a condition for identifying new sessions: if 'duration' is null or greater than 1800 seconds (30 minutes)
condition_23 = (F.col('duration').isNull() | (F.col('duration') > 1800))

# Define a window specification ordering by 'cdp_id' and 'timestamp'
window_spec_7 = Window.orderBy('cdp_id', 'timestamp')

# Generate a unique session ID by summing up the instances that meet the condition (new session detected)
# The result is cumulative, creating a unique session ID for each detected session.
result_df = result_df.withColumn('session',
                                 F.when(condition_23, 
                                        F.sum(F.when(condition_23, 1).otherwise(0)).over(window_spec_7))
                                 .otherwise(
                                     F.sum(F.when(condition_23, 1).otherwise(0)).over(window_spec_7)))


In [ ]:
# result_df.agg(F.count('session'),
#               F.countDistinct('session'),
#               F.max('session'))

### Remove unnecessary actions

In [7]:
# Filter out rows where the 'action' column is null
result_df = result_df.filter(col('action').isNotNull())

# Identify sessions that contain 'error', 'service', or 'landing' actions
lst_session_exclude = result_df.filter(col('action').isin(['error', 'service', 'landing'])).groupby('session').count()

# Perform a left join between the original DataFrame and the sessions to exclude based on the 'session' column
result_df_1 = result_df.join(
    other=lst_session_exclude,
    how='left',
    on='session')

# Filter out rows that belong to sessions identified for exclusion by checking if 'count' is null
result_df_1 = result_df_1.filter(col('count').isNull())

# Drop the 'count' column used for session exclusion
result_df_1 = result_df_1.drop("count")


### Save data

In [8]:

(
    result_df_1
    # .withColumn(
    #     "d", F.lit("2024-01-01")
    # )  # add thêm column partition nếu chưa có
    # .repartition(
    #     92
    # )  # repartition để lưu nhanh hơn và đọc lên nhanh hơn(nếu data sẽ được sort theo columns nào đó, thứ tự khi đọc lên sẽ không được đảm bảo)
    .write.partitionBy("d")  # partition theo column export_date
    .mode("overwrite")
    .option("partitionOverwriteMode", "dynamic")  # cho phép lưu đè
    .parquet(
        f"/data/fpt/ftel/cads/dep_solution/user/{user}/fptshop_sample_2024-08-15-action-full-1.parquet"
    )
)


## Version 2 : aggregate consecutive actions

In [24]:
result_df = spark.read.parquet(f"/data/fpt/ftel/cads/dep_solution/user/{user}/fptshop_sample_2024-08-15-action-full-1.parquet")

In [25]:
result_df

session,timestamp,ip,cdp_id,event,type,user_agent,item_title,item_id,category_id,category_name,item_cate,subcategory_id,subcategory_name,path,referrer,search,title,url,campaign_content,campaign_medium,campaign_name,campaign_source,campaign_term,path_1,path_2,path_3,path_4,path_5,isInstallment,total_isInstallment,total_isInstallment_1,action,total_isTrack,total_isTrack_1,total_isTrack_2,product,type_product,duration,d
86,2024-08-13 23:14:27,1.52.133.140,00001972-bfe0-4c7b-933d-e748223a5b29,null,page,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36",null,null,[],null,null,null,null,/file/84k9cp7k133x,https://www.fshare.vn/folder/PVZ1W7XT2I3N?token=1723565575,?token=1723565665,CAMBRIDGE IELTS 9 AUDIO.zip - Fshare,https://www.fshare.vn/file/84K9CP7K133X?token=1723565665,null,null,null,null,null,file,84k9cp7k133x,null,null,null,0,0,0,viewPage,12,318220,12,CAMBRIDGE IELTS 9 AUDIO.zip - Fshare,file,null,2024-08-13
86,2024-08-13 23:14:28,1.52.133.140,00001972-bfe0-4c7b-933d-e748223a5b29,null,page,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36",CAMBRIDGE IELTS 9 AUDIO.zip,84K9CP7K133X,[],null,file,null,null,/file/84k9cp7k133x,https://www.fshare.vn/folder/PVZ1W7XT2I3N?token=1723565575,?token=1723565665,CAMBRIDGE IELTS 9 AUDIO.zip - Fshare,https://www.fshare.vn/file/84K9CP7K133X?token=1723565665,null,null,null,null,null,file,84k9cp7k133x,null,null,null,0,0,0,viewPage,12,318220,12,CAMBRIDGE IELTS 9 AUDIO.zip - Fshare,file,1,2024-08-13
323,2024-08-13 22:16:33,14.184.126.221,00006d47-9a8c-4f23-a31c-bd670b46e755,null,page,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36",null,null,[],null,null,null,null,/file/5r8jrw16eava,https://gta5vn.net/,?token=1723562192,Grand_Theft_Auto_V_2824.rar - Fshare,https://www.fshare.vn/file/5R8JRW16EAVA?token=1723562192,null,null,null,null,null,file,5r8jrw16eava,null,null,null,0,0,0,viewPage,834,318220,834,Grand_Theft_Auto_V_2824.rar - Fshare,file,null,2024-08-13
323,2024-08-13 22:16:34,14.184.126.221,00006d47-9a8c-4f23-a31c-bd670b46e755,null,page,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36",Grand_Theft_Auto_V_2824.rar,5R8JRW16EAVA,[],null,file,null,null,/file/5r8jrw16eava,https://gta5vn.net/,?token=1723562192,Grand_Theft_Auto_V_2824.rar - Fshare,https://www.fshare.vn/file/5R8JRW16EAVA?token=1723562192,null,null,null,null,null,file,5r8jrw16eava,null,null,null,0,0,0,viewPage,834,318220,834,Grand_Theft_Auto_V_2824.rar - Fshare,file,1,2024-08-13
323,2024-08-13 22:16:45,14.184.126.221,00006d47-9a8c-4f23-a31c-bd670b46e755,null,page,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36",null,null,[],null,null,null,null,/file/5r8jrw16eava,https://gta5vn.net/,?token=1723562204,Grand_Theft_Auto_V_2824.rar - Fshare,https://www.fshare.vn/file/5R8JRW16EAVA?token=1723562204,null,null,null,null,null,file,5r8jrw16eava,null,null,null,0,0,0,viewPage,834,318220,834,Grand_Theft_Auto_V_2824.rar - Fshare,file,11,2024-08-13
323,2024-08-13 22:16:46,14.184.126.221,00006d47-9a8c-4f23-a31c-bd670b46e755,null,page,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36",Grand_Theft_Auto_V_2824.rar,5R8JRW16EAVA,[],null,file,null,null,/file/5r8jrw16eava,https://gta5vn.net/,?token=1723562204,Grand_Theft_Auto_V_2824.rar - Fshare,https://www.fshare.vn/file/5R8JRW16EAVA?token=1723562204,null,null,null,null,null,file,5r8jrw16eava,null,null,null,0,0,0,viewPage,834,318220,834,Grand_Theft_Auto_V_2824.rar - Fshare,file,1,2024-08-13
323,2024-08-13 22:38:53,14.184.126.221,00006d47-9a8c-4f23-a31c-bd670b46e755,null,page,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36",null,null,[],null,null,null,null,/file/5r8jrw16ea

In [27]:
# Create a Window specification to define the order of rows
windowSpec = Window().partitionBy('session').orderBy("timestamp")

# Add a new column "prev_col2" with the value of the previous "col2"
result_df = result_df.withColumn("previous_action", F.lag("action").over(windowSpec))

result_df = result_df.filter(
    (col('action') != col('previous_action')) | (col('action').isNull()) | (col('previous_action').isNull()))

### Recalculate duration and session

In [28]:
# Convert the timestamp column to a date type
# df = df.withColumn("timestamp", col("timestamp").cast("date"))

# Define a window specification to partition by 'cdp_id' and order by the timestamp
window_spec_5 = Window().partitionBy("session").orderBy("timestamp")

# Calculate the previous timestamp within each group
result_df = result_df.withColumn("prev_timestamp", lag("timestamp").over(window_spec_5))

# Calculate the percentage change
result_df = result_df.withColumn("duration",
                                 (col("timestamp").cast("long") - col("prev_timestamp").cast("long")))

# Drop the temporary column used for lag
result_df = result_df.drop("prev_timestamp")


In [29]:
windowSpec = Window().partitionBy('session').orderBy('timestamp')
result_df = result_df.withColumn('rank', when(condition=(col('action').isin(['payment', 'installment'])),
                                              value=F.rank().over(windowSpec)))

result_df = result_df.withColumn('action_previous_1',
                                 F.lag(col('action')).over(windowSpec))
result_df = result_df.withColumn('action_previous_2',
                                 F.lag(col('action'), 2).over(windowSpec))
result_df = result_df.withColumn('action_previous_3',
                                 F.lag(col('action'), 3).over(windowSpec))
result_df = result_df.withColumn('action_previous_4',
                                 F.lag(col('action'), 4).over(windowSpec))
result_df = result_df.withColumn('action_previous_5',
                                 F.lag(col('action'), 5).over(windowSpec))
result_df = result_df.withColumn('action_previous_6',
                                 F.lag(col('action'), 6).over(windowSpec))
result_df = result_df.withColumn('action_previous_7',
                                 F.lag(col('action'), 7).over(windowSpec))
result_df = result_df.withColumn('action_previous_8',
                                 F.lag(col('action'), 8).over(windowSpec))
result_df = result_df.withColumn('action_previous_9',
                                 F.lag(col('action'), 9).over(windowSpec))
result_df = result_df.withColumn('action_previous_10',
                                 F.lag(col('action'), 10).over(windowSpec))

result_df = result_df.withColumn('action_following_1',
                                 F.lag(col('action'), -1).over(windowSpec))
result_df = result_df.withColumn('action_following_2',
                                 F.lag(col('action'), -2).over(windowSpec))
result_df = result_df.withColumn('action_following_3',
                                 F.lag(col('action'), -3).over(windowSpec))
result_df = result_df.withColumn('action_following_4',
                                 F.lag(col('action'), -4).over(windowSpec))
result_df = result_df.withColumn('action_following_5',
                                 F.lag(col('action'), -5).over(windowSpec))
result_df = result_df.withColumn('action_following_6',
                                 F.lag(col('action'), -6).over(windowSpec))
result_df = result_df.withColumn('action_following_7',
                                 F.lag(col('action'), -7).over(windowSpec))
result_df = result_df.withColumn('action_following_8',
                                 F.lag(col('action'), -8).over(windowSpec))
result_df = result_df.withColumn('action_following_9',
                                 F.lag(col('action'), -9).over(windowSpec))
result_df = result_df.withColumn('action_following_10',
                                 F.lag(col('action'), -10).over(windowSpec))
result_df = result_df.withColumn('action_following_11',
                                 F.lag(col('action'), -11).over(windowSpec))

In [30]:
columns_to_drop = [
    'ip', 'anon_id', 'user_id', 'type', 'messageId', 'total_isInstallment_1',
    'total_isInstallment', 'isInstallment', 'total_isTrack_2', 'total_isTrack_1',
    'total_isTrack', 'path_5', 'app_name', 'f_partner', 'name', 'customer_name',
    'address', 'email', 'phone', 'object_type', 'object_thumb', 'object_id',
    'brand_id', 'app_id', 'subcategory_name', 'subcategory_id', 'item_cate',
    'category_name', 'category_id'
]

# Drop the specified columns
result_df = result_df.drop(*columns_to_drop)

In [31]:
result_df = result_df.withColumn('Purchase',
                                 when(col('action').isin(['payment', 'installment', 'purchase']), 1).otherwise(0))
# Create a new column 'isPurchase' based on the conditions
condition_30 = (F.sum('Purchase').over(Window().partitionBy('session')) > 0)
result_df = result_df.withColumn(
    'isPurchase',
    when(
        condition=condition_30
        , value=1
    ).otherwise(0)
)

In [32]:
condition_31 = col('action').isin(['viewNews', 'homePage', 'checkPromo'])
new_value_31 = 'Awareness'

condition_32 = col('action').isin(
    ['viewPage', 'viewDetail', 'findProduct', 'lookup', 'viewOldProduct', 'support', 'compareProduct', 'upgrade'])
new_value_32 = 'Consideration'

condition_33 = col('action').isin(['installment', 'addToCart', 'payment', 'purchase', 'checkCart'])
new_value_33 = 'Decision'

condition_34 = col('action').isin(['trackingOrder', 'checkWarranty'])
new_value_34 = 'Consumption'

condition_35 = col('action').isin(['checkAccount', 'checkLoyalty'])
new_value_35 = 'Loyalty'

result_df = result_df.withColumn("phase",
                                 when(condition_31, new_value_31)
                                 .when(condition_32, new_value_32)
                                 .when(condition_33, new_value_33)
                                 .when(condition_34, new_value_34)
                                 .when(condition_35, new_value_35)
                                 .otherwise(None)  # Keep the original value if none of the conditions match
                                 )

In [33]:
lst_session_exclude = result_df.filter(col('phase').isNull()).groupby('session').count()

In [34]:
result_df = result_df.join(
    other=lst_session_exclude,
    how='left',
    on='session')

result_df = result_df.filter(col('count').isNull())

result_df.filter(col('phase').isNull()).count()

0

In [35]:
# Convert the values in the 'product' column to lowercase
result_df = result_df.withColumn('product', F.lower('product'))
# Split the values in the 'product' column and extract the first element
result_df = result_df.withColumn('brand', F.split('product', ' ').getItem(0))

result_df = result_df.withColumn('brand',
                                 when(col('product').contains('iphone'), 'apple')
                                 .when(col('product').contains('macbook'), 'apple')
                                 .when(col('product').contains('ipad'), 'apple')
                                 .when(col('product').contains('imac'), 'apple')
                                 .when(col('product').contains('apple'), 'apple')
                                 .when(col('product').contains('samsung'), 'samsung')
                                 .when(col('product').contains('xiaomi'), 'xiaomi')
                                 .when(col('product').contains('redmi'), 'xiaomi')
                                 .when(col('product').contains('asus'), 'asus')
                                 .when(col('product').contains('lenovo'), 'lenovo')
                                 .when(col('product').contains('oppo'), 'oppo')
                                 .when(col('product').contains('acer'), 'acer')
                                 .when(col('product').contains('msi'), 'msi')
                                 .when(col('product').contains('hp'), 'hp')
                                 .when(col('product').contains('dell'), 'dell')
                                 .when(col('product').contains('honor'), 'honor')
                                 .when(col('product').contains('realme'), 'realme')
                                 .when(col('product').contains('nokia'), 'nokia')
                                 .when(col('product').contains('huawei'), 'huawei')
                                 .when(col('product').contains('huwei'), 'huawei')
                                 .when(col('product').contains('vivo'), 'vivo')
                                 .when(col('product').contains('gigabyte'), 'gigabyte')
                                 .when(col('product').contains('masstel'), 'masstel')
                                 .otherwise(col('brand'))
                                 )

In [36]:
result_df = result_df.withColumn('source',
                                 when(col('referrer') == "", 'universe')
                                 .when(col('referrer').contains('google'), 'google')
                                 .when(col('referrer').contains('facebook'), 'facebook')
                                 .when(col('referrer').contains('youtube'), 'youtube')
                                 .when(col('referrer').contains('fpt'), 'fpt')
                                 .when(col('referrer').contains('bing'), 'bing')
                                 .when(col('referrer').contains('cdn'), 'cdn')
                                 .when(col('referrer').contains('yahoo'), 'yahoo')
                                 .when(col('referrer').contains('coccoc'), 'coccoc')
                                 .when(col('referrer').contains('tiktok'), 'tiktok')
                                 .when(col('referrer').contains('zalo'), 'zalo')
                                 .when(col('referrer').contains('eclick'), 'eclick')
                                 .when(col('referrer').contains('24h'), '24h')
                                 .when(col('referrer').contains('dantri'), 'dantri')
                                 .when(col('referrer').contains('kenh14'), 'kenh14')
                                 .when(col('referrer').contains('msi'), 'msi')
                                 .when(col('referrer').contains('msn'), 'msn')
                                 .when(col('referrer').contains('thanhnien'), 'thanhnien')
                                 .otherwise('other')
                                 )

In [37]:
(
    result_df
    .write.partitionBy("d")  # partition theo column export_date
    .mode("overwrite")
    .option("partitionOverwriteMode", "dynamic")  # cho phép lưu đè
    .parquet(
        f"/data/fpt/ftel/cads/dep_solution/user/{user}/fptshop_sample_2024-08-15-action-full-2.parquet"
    )
)

In [38]:
# Create a Window specification to define the order of rows
windowSpec_100 = Window().partitionBy('session').orderBy("timestamp")
# Add a new column "prev_col2" with the value of the previous "col2"
result_df = result_df.withColumn("previous_phase", F.lag("phase").over(windowSpec_100))

result_df = result_df.filter(
    (col('phase') != col('previous_phase')) | (col('phase').isNull()) | (col('previous_phase').isNull()))

In [39]:
# Convert the timestamp column to a date type
# df = df.withColumn("timestamp", col("timestamp").cast("date"))

# Define a window specification to partition by 'cdp_id' and order by the timestamp
window_spec_5 = Window().partitionBy("cdp_id").orderBy("timestamp")

# Calculate the previous timestamp within each group
result_df = result_df.withColumn("prev_timestamp", lag("timestamp").over(window_spec_5))

# Calculate the percentage change
result_df = result_df.withColumn("duration",
                                 (col("timestamp").cast("long") - col("prev_timestamp").cast("long")))

# Drop the temporary column used for lag
result_df = result_df.drop("prev_timestamp")


In [40]:
windowSpec = Window().partitionBy('session').orderBy('timestamp')
# result_df = result_df.withColumn('rank', when(condition=(col('action').isin(['payment', 'installment'])),
#                                               value=F.rank().over(windowSpec)))

result_df = result_df.withColumn('phase_previous_1',
                                 F.lag(col('phase')).over(windowSpec))
result_df = result_df.withColumn('phase_previous_2',
                                 F.lag(col('phase'), 2).over(windowSpec))
result_df = result_df.withColumn('phase_previous_3',
                                 F.lag(col('phase'), 3).over(windowSpec))
result_df = result_df.withColumn('phase_previous_4',
                                 F.lag(col('phase'), 4).over(windowSpec))
result_df = result_df.withColumn('phase_previous_5',
                                 F.lag(col('phase'), 5).over(windowSpec))
result_df = result_df.withColumn('phase_previous_6',
                                 F.lag(col('phase'), 6).over(windowSpec))
result_df = result_df.withColumn('phase_previous_7',
                                 F.lag(col('phase'), 7).over(windowSpec))
result_df = result_df.withColumn('phase_previous_8',
                                 F.lag(col('phase'), 8).over(windowSpec))
result_df = result_df.withColumn('phase_previous_9',
                                 F.lag(col('phase'), 9).over(windowSpec))
result_df = result_df.withColumn('phase_previous_10',
                                 F.lag(col('phase'), 10).over(windowSpec))

result_df = result_df.withColumn('phase_following_1',
                                 F.lag(col('phase'), -1).over(windowSpec))
result_df = result_df.withColumn('phase_following_2',
                                 F.lag(col('phase'), -2).over(windowSpec))
result_df = result_df.withColumn('phase_following_3',
                                 F.lag(col('phase'), -3).over(windowSpec))
result_df = result_df.withColumn('phase_following_4',
                                 F.lag(col('phase'), -4).over(windowSpec))
result_df = result_df.withColumn('phase_following_5',
                                 F.lag(col('phase'), -5).over(windowSpec))
result_df = result_df.withColumn('phase_following_6',
                                 F.lag(col('phase'), -6).over(windowSpec))
result_df = result_df.withColumn('phase_following_7',
                                 F.lag(col('phase'), -7).over(windowSpec))
result_df = result_df.withColumn('phase_following_8',
                                 F.lag(col('phase'), -8).over(windowSpec))
result_df = result_df.withColumn('phase_following_9',
                                 F.lag(col('phase'), -9).over(windowSpec))
result_df = result_df.withColumn('phase_following_10',
                                 F.lag(col('phase'), -10).over(windowSpec))
result_df = result_df.withColumn('phase_following_11',
                                 F.lag(col('phase'), -11).over(windowSpec))

In [41]:

(
    result_df
    .write.partitionBy("d")  # partition theo column export_date
    .mode("overwrite")
    .option("partitionOverwriteMode", "dynamic")  # cho phép lưu đè
    .parquet(
        f"/data/fpt/ftel/cads/dep_solution/user/{user}/fptshop_sample_2024-08-15-action-full-3.parquet"
    )
)


In [43]:
spark.read.parquet(f'/data/fpt/ftel/cads/dep_solution/user/{user}/fptshop_sample_2024-08-15-action-full-3.parquet')

session,timestamp,cdp_id,event,user_agent,item_title,item_id,path,referrer,search,title,url,campaign_content,campaign_medium,campaign_name,campaign_source,campaign_term,path_1,path_2,path_3,path_4,action,product,type_product,duration,previous_action,rank,action_previous_1,action_previous_2,action_previous_3,action_previous_4,action_previous_5,action_previous_6,action_previous_7,action_previous_8,action_previous_9,action_previous_10,action_following_1,action_following_2,action_following_3,action_following_4,action_following_5,action_following_6,action_following_7,action_following_8,action_following_9,action_following_10,action_following_11,Purchase,isPurchase,phase,count,brand,source,previous_phase,phase_previous_1,phase_previous_2,phase_previous_3,phase_previous_4,phase_previous_5,phase_previous_6,phase_previous_7,phase_previous_8,phase_previous_9,phase_previous_10,phase_following_1,phase_following_2,phase_following_3,phase_following_4,phase_following_5,phase_following_6,phase_following_7,phase_following_8,phase_following_9,phase_following_10,phase_following_11,d
75758,2024-08-14 11:10:25,0070931f-8546-41c3-8ed5-f42eaec87d56,null,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4 Safari/605.1.15 (Applebot/0.1; +http://www.apple.com/go/applebot)",null,null,/file/ckik3ej6z7s1,,?token=1723607806,Không tìm thấy - Fshare,https://www.fshare.vn/file/CKIK3EJ6Z7S1?token=1723607806,null,null,null,null,null,file,ckik3ej6z7s1,null,null,findProduct,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,Consideration,null,null,universe,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-08-14
86490,2024-08-14 12:52:47,0080a548-ff3f-4329-a8a4-80069eec93a5,null,"Mozilla/5.0 (Linux; Android 10; K) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Mobile Safari/537.36",null,null,/khuyen-mai/dac-quyen-1k,https://id.zalo.me/,?zarsrc=33&utm_source=zalo&utm_medium=zalo&utm_campaign=zalo&gidzl=UPdp7gWkkovtZlSUeZEyA76AznouPxmZQzdo4Erdl7KvsV52lpRjVJsEgKpYDRLoOOtv4JFdZdq7hII-AW,Nhà thuốc Long Châu - Hệ thống chuỗi nhà thuốc lớn,https://nhathuoclongchau.com.vn/khuyen-mai/dac-quyen-1k?zarsrc=33&utm_source=zalo&utm_medium=zalo&utm_campaign=zalo&gidzl=UPdp7gWkkovtZlSUeZEyA76AznouPxmZQzdo4Erdl7KvsV52lpRjVJsEgKpYDRLoOOtv4JFdZdq...,null,zalo,zalo,zalo,null,khuyen-mai,dac-quyen-1k,null,null,checkPromo,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,Awareness,null,null,zalo,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-08-14
134116,2024-08-14 06:50:43,00c9395b-8845-45b5-b048-e42ee4bf68ff,null,"Mozilla/5.0 (Linux; Android 13; SM-A715F Build/TP1A.220624.014; wv) AppleWebKit/537.36 (KHTML, like Gecko) Version/4.0 Chrome/127.0.6533.103 Mobile Safari/537.36 open_news open_news_u_s/5411",null,null,/khuyen-mai/ca-nha-khoe-tre-den-truong,https://www.pangleglobal.com/,?utm_source=Tiktok&utm_medium=CPC&utm_campaign=Main-T8-2024&lpt=1&ttclid=E.C.P.CssB_QapTX1J7Jz5uoTsgL4MZ29V93YJ2HIuldKONm53zLeIRX9m87wEkv1RMrkb6PI8GZTA-M4DW-muJYdwyzSseb66Q-luJBDh92N31Hskcuf6O8VnEj...,CẢ NHÀ KHỎE - TRẺ ĐI HỌC - GIẢM ĐẾN 35% & MUA 1 TẶNG 1,https://nhathuoclongchau.com.vn/khuyen-mai/ca-nha-khoe-tre-den-truong?utm_source=Tiktok&utm_medium=CPC&utm_campaign=Main-T8-2024&lpt=1&ttclid=E.C.P.CssB_QapTX1J7Jz5uoTsgL4MZ29V93YJ2HIuldKONm53zLeIR...,null,CPC,Main-T8-2024,Tiktok,null,khuyen-mai,ca-nha-khoe-tre-den-truong,null,null,checkPromo,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,Awareness,null,null,other,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-08-14
255860,2024-08-14 16:09:39,0182d713-faa

In [44]:
spark.read.parquet(f'/data/fpt/ftel/cads/dep_solution/user/{user}/ggg/raw/fptshop_sample_2024-08-15-action-full-3.parquet')

AnalysisException: Path does not exist: hdfs://hdfs-cluster.datalake.bigdata.local:8020/data/fpt/ftel/cads/dep_solution/user/hoanglx9/ggg/raw/fptshop_sample_2024-08-15-action-full-3.parquet;